## Descrição
Este notebook implementa o pipeline **ELT (Extract, Load, Transform)** para integração
das bases do Banco de Preços em Saúde (BPS) referentes aos anos de **2023, 2024 e 2025**.

Diferente do pipeline **ETL**, neste fluxo os dados são extraídos e carregados no banco
de dados PostgreSQL **sem nenhuma transformação prévia**. Toda a limpeza, padronização e
modelagem dimensional ocorre **dentro do próprio banco**, por meio do projeto **dbt**
(`transformacao_bps/`).

## Resultado deste notebook
Ao final da execução, o banco `bps_dw` terá três tabelas com os dados brutos:
- `raw_bps_2023` — base bruta do ano de 2023
- `raw_bps_2024` — base bruta do ano de 2024
- `raw_bps_2025` — base bruta do ano de 2025

Essas tabelas servirão como **fonte (source)** para os models do dbt, que produzirão
o Esquema Estrela final.



In [2]:
# SEÇÃO 1 — Instalação de dependências e imports

# O flag -q suprime o output verbose do pip, deixando o notebook mais limpo
%pip install pandas sqlalchemy psycopg2-binary python-dotenv -q

# ── Imports ──────────────────────────────────────────────────
import pandas as pd                        # Leitura dos CSVs brutos
from sqlalchemy import create_engine, text # Conexão com PostgreSQL
from dotenv import load_dotenv             # Leitura do .env
import os                                  # Acesso às variáveis de ambiente

print("✅ Bibliotecas instaladas e importadas com sucesso!")


Note: you may need to restart the kernel to use updated packages.
✅ Bibliotecas instaladas e importadas com sucesso!


## 2. Conexão com o PostgreSQL

O arquivo `.env` esperado tem a seguinte estrutura:

```
DB_USER=postgres
DB_PASSWORD=sua_senha
DB_HOST=localhost
DB_PORT=5432
DB_NAME=bps_dw
```

A conexão é estabelecida **logo no início** do ELT — diferente do ETL, onde ela ocorria
na Seção 4. Aqui isso é necessário porque a carga no banco acontece imediatamente após
a leitura de cada CSV, sem etapas intermediárias em memória.


In [3]:
# SEÇÃO 2 — Conexão com o PostgreSQL

load_dotenv('../.env')

DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST     = os.getenv('DB_HOST')
DB_PORT     = os.getenv('DB_PORT')
DB_NAME     = os.getenv('DB_NAME')

DATABASE_URL = (
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_NAME}?client_encoding=utf8'
)

print("Tentando conectar ao PostgreSQL...")

try:
    engine = create_engine(DATABASE_URL)
    with engine.connect() as conn:
        versao = conn.execute(text("SELECT version();")).scalar()
    print(f"✅ Conexão estabelecida com {DB_NAME}!")
except Exception as e:
    print(f"❌ Erro ao conectar: {e}")


Tentando conectar ao PostgreSQL...
✅ Conexão estabelecida com bps_dw!


## 3. Extract + Load (EL)

Esta é a etapa central do pipeline ELT. Para cada arquivo CSV, realizamos:

1. **Extract** — lê o CSV com `pd.read_csv()`, respeitando o padrão das bases do
   governo brasileiro: separador `;` e encoding `latin-1`
2. **Load** — despeja o DataFrame **inteiro e sem qualquer transformação** no
   PostgreSQL usando `df.to_sql()`

> ⚠️ **Importante:** nenhuma limpeza é realizada aqui. CNPJs com pontuação, datas em
> formato string, preços com vírgula decimal, espaços em branco — tudo é carregado
> **exatamente como está no CSV original**. Essa é a característica fundamental que
> distingue o ELT do ETL.


In [ ]:
# SEÇÃO 3 — EXTRACT + LOAD

arquivos_brutos = {
    'raw_bps_2023': '../data/raw/2023.csv',
    'raw_bps_2024': '../data/raw/2024.csv',
    'raw_bps_2025': '../data/raw/2025.csv',
}

print("Iniciando carga dos dados BRUTOS no PostgreSQL...\n")

for nome_tabela, caminho_arquivo in arquivos_brutos.items():
    try:
        print(f"Lendo: {caminho_arquivo}")
        df_raw = pd.read_csv(
            caminho_arquivo,
            sep=';',
            encoding='latin-1',
            on_bad_lines='skip'
        )

        print(f"→ {len(df_raw):,} linhas × {len(df_raw.columns)} colunas")
        print(f"Carregando na tabela: {nome_tabela}...")

        # O Load — dados brutos, sem transformação
        df_raw.to_sql(
            name=nome_tabela,
            con=engine,
            if_exists='replace',
            index=False,
            chunksize=5000,
            method='multi'
        )

        print(f"✅ Tabela '{nome_tabela}' carregada com sucesso.\n")

    except FileNotFoundError:
        print(f"❌ Arquivo não encontrado: {caminho_arquivo}\n")
    except Exception as e:
        print(f"❌ Erro ao processar {caminho_arquivo}: {e}\n")

print("Processo de EL (Extract + Load) concluído.")


Iniciando carga dos dados BRUTOS no PostgreSQL...

Lendo: data/2023.csv
→ 31,992 linhas × 25 colunas
Carregando na tabela: raw_bps_2023...
✅ Tabela 'raw_bps_2023' carregada com sucesso.

Lendo: data/2024.csv
→ 26,258 linhas × 25 colunas
Carregando na tabela: raw_bps_2024...
✅ Tabela 'raw_bps_2024' carregada com sucesso.

Lendo: data/2025.csv
→ 26,215 linhas × 25 colunas
Carregando na tabela: raw_bps_2025...
✅ Tabela 'raw_bps_2025' carregada com sucesso.

Processo de EL (Extract + Load) concluído.


## 4. Verificação da carga

Após o Load, verificamos se as tabelas foram criadas corretamente com o número
esperado de registros. Fazemos dois tipos de verificação:

1. **Contagem de registros** — `SELECT COUNT(*)` para cada tabela, confirmando
   que nenhuma linha foi perdida durante a inserção
2. **Preview dos dados brutos** — exibimos as primeiras linhas de cada tabela
   para checar visualmente que os dados estão conforme o CSV original —
   inclusive com toda a "sujeira" característica (formato de data BR, preços
   com vírgula, CNPJs com pontuação, etc.)


In [8]:
# SEÇÃO 4 — VERIFICAÇÃO DA CARGA

print("Verificando a carga no PostgreSQL...\n")

with engine.connect() as conn:
    for nome_tabela in arquivos_brutos.keys():
        resultado = conn.execute(text(f"SELECT COUNT(*) FROM {nome_tabela};"))
        total = resultado.scalar()
        print(f"   📊 {nome_tabela}: {total:,} linhas")

print("\n✅ Todas as tabelas brutas estão disponíveis no banco.")


Verificando a carga no PostgreSQL...

   📊 raw_bps_2023: 31,992 linhas
   📊 raw_bps_2024: 26,258 linhas
   📊 raw_bps_2025: 26,215 linhas

✅ Todas as tabelas brutas estão disponíveis no banco.


In [10]:
# Preview das tabelas brutas — dados exatamente como vieram do CSV

for nome_tabela in arquivos_brutos.keys():
    print(f"Preview: {nome_tabela}")
    preview = pd.read_sql(f"SELECT * FROM {nome_tabela} LIMIT 3;", con=engine)
    display(preview)


Preview: raw_bps_2023


,ano_compra,nome_instituicao,esfera,cnpj_instituicao,municipio_instituicao,uf,compra,insercao,codigo_br,descricao_catmat,...,capacidade,unidade_medida,unidade_fornecimento_capacidade,cnpj_fornecedor,fornecedor,cnpj_fabricante,fabricante,qtd_itens_comprados,preco_unitario,preco_total
0,2023,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18.478.187/0001-07,MARABA,PA,01/01/2023,15/03/2024,423975,"PIPETA, TIPO:PASTEUR, CAPACIDADE:3 ML, MATERIA...",...,NaN,NaN,UNIDADE,05.323.167/0001-07,CIRUBEL COMERCIO E REPRESENTACOES DE PRODUTOS ...,57.893.521/0001-32,PERFITECNICA PERFIS TECNICOS DE BORRACHA LTDA,36,45.00,1620.0
1,2023,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18.478.187/0001-07,MARABA,PA,01/01/2023,15/03/2024,457503,"CORANTE, TIPO :PARDO DE BISMARCK, CARACTERÃST...",...,NaN,NaN,UNIDADE,07.944.100/0001-15,PROC9 INDUSTRIA QUIMICA LTDA,07.944.100/0001-15,PROC9 INDUSTRIA QUIMICA LTDA,4,190.00,760.0
2,2023,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18.478.187/0001-07,MARABA,PA,01/01/2023,15/03/2024,382447,"REAGENTE PARA DIAGNÃSTICO CLÃNICO 5, TIPO:AL...",...,10.0,ML,FRASCO 10.00 ML,05.048.534/0001-01,NORTEMED DISTRIBUIDORA DE PRODUTOS MEDICOS LTDA,50.657.402/0001-31,EBRAM PRODUTOS LABORATORIAIS LTDA,20,33.38,667.6


Preview: raw_bps_2024


,ano_compra,nome_instituicao,esfera,cnpj_instituicao,municipio_instituicao,uf,compra,insercao,codigo_br,descricao_catmat,...,capacidade,unidade_medida,unidade_fornecimento_capacidade,cnpj_fornecedor,fornecedor,cnpj_fabricante,fabricante,qtd_itens_comprados,preco_unitario,preco_total
0,2024,FUNDO MUNICIPAL DE SAUDE,MUNICIPAL,11.120.699/0001-40,MURICI,AL,01/01/2024,09/08/2024,268375,"ACICLOVIR, DOSAGEM:50 MG/G, USO:CREME",...,10.0,G,BISNAGA 10.00 G,00.236.193/0001-84,CIRURGICA RECIFE COMERCIO E REPRESENTACOES LTDA,73.856.593/0001-66,"PRATI, DONADUZZI & CIA LTDA",900,5.24,4716.0
1,2024,FUNDO MUNICIPAL DE SAUDE,MUNICIPAL,21.817.418/0001-66,URUPA,RO,01/01/2024,21/08/2025,267203,"DIPIRONA SÃDICA, DOSAGEM:500 MG",...,NaN,NaN,COMPRIMIDO,26.419.311/0001-83,LUMANN DISTRIBUIDORA DE MEDICAMENTOS LTDA,57.507.378/0001-01,EMS S/A,25000,0.17,4250.0
2,2024,FUNDO MUNICIPAL DE SAUDE,MUNICIPAL,11.120.699/0001-40,MURICI,AL,01/01/2024,09/08/2024,268370,"ACICLOVIR, DOSAGEM:200 MG",...,NaN,NaN,COMPRIMIDO,00.236.193/0001-84,CIRURGICA RECIFE COMERCIO E REPRESENTACOES LTDA,73.856.593/0001-66,"PRATI, DONADUZZI & CIA LTDA",600,0.33,198.0


Preview: raw_bps_2025


,ano_compra,nome_instituicao,esfera,cnpj_instituicao,municipio_instituicao,uf,compra,insercao,codigo_br,descricao_catmat,...,capacidade,unidade_medida,unidade_fornecimento_capacidade,cnpj_fornecedor,fornecedor,cnpj_fabricante,fabricante,qtd_itens_comprados,preco_unitario,preco_total
0,2025,FUNDO MUNICIPAL DE SAUDE,MUNICIPAL,11.328.684/0001-71,SAO FRANCISCO DO GUAPORE,RO,01/01/2025,01/09/2025,342134,"HIDROCORTISONA, COMPOSIÃÃO:SAL SUCCINATO SÃ...",...,NaN,NaN,FRASCO-AMPOLA,58.430.828/0001-60,BLAU FARMACEUTICA S.A.,58.430.828/0001-60,BLAU FARMACEUTICA S.A.,200,11.91,2382.0
1,2025,FUNDO MUNICIPAL DE SAUDE,MUNICIPAL,11.328.684/0001-71,SAO FRANCISCO DO GUAPORE,RO,01/01/2025,01/09/2025,267772,"PROPRANOLOL CLORIDRATO, DOSAGEM:40 MG",...,NaN,NaN,COMPRIMIDO,19.791.813/0001-75,LABORATORIOS OSORIO DE MORAES LTDA,19.791.813/0001-75,LABORATORIOS OSORIO DE MORAES LTDA,5000,0.05,250.0
2,2025,FUNDO MUNICIPAL DE SAUDE,MUNICIPAL,11.328.684/0001-71,SAO FRANCISCO DO GUAPORE,RO,01/01/2025,01/09/2025,267310,"METOCLOPRAMIDA CLORIDRATO, DOSAGEM:5 MG/ML, AP...",...,2.0,ML,AMPOLA 2.00 ML,01.571.702/0001-98,HALEX ISTAR INDUSTRIA FARMACEUTICA SA,01.571.702/0001-98,HALEX ISTAR INDUSTRIA FARMACEUTICA SA,300,0.81,243.0


## 5. Próximos passos — Transformações com dbt

A etapa **EL** está concluída. As três tabelas `raw_bps_*` agora estão no
PostgreSQL com os dados brutos, prontas para serem consumidas pelo dbt.

A partir daqui, toda a lógica de transformação migra para o projeto dbt em
`transformacao_bps/`, organizado em duas camadas:

| Camada | Localização | O que faz |
|---|---|---|
| **staging** | `transformacao_bps/models/staging/` | Limpa tipos, CNPJs, datas e preços; unifica os 3 anos em uma view intermediária |
| **marts** | `transformacao_bps/models/marts/` | Constrói o Star Schema: `dim_produto`, `dim_instituicao`, `dim_fornecedor`, `dim_tempo` e `fato_compras` |

Para executar o dbt após rodar este notebook:

```bash
cd transformacao_bps
dbt run      # executa todos os models
dbt test     # roda os testes de qualidade
dbt docs generate && dbt docs serve   # abre a documentação interativa
```
